# The Cooperative Grid — TinyML Experiment Notebook
### Evidence-driven forecasting, four-node inference, and deployment preparation

**Purpose.** This notebook is the experimental bridge between the approved SLDC Odisha workbook, a compact forecasting model, the four ESP32 node MVP, and the Raspberry Pi coordinator.

**Evidence labels used throughout**
- **Workbook-derived:** directly read from the supplied Excel file.
- **Configured:** selected by the project team and must be justified.
- **Simulated:** generated for pipeline testing; not real feeder measurements.
- **Not yet validated:** requires hardware, electrical-network, or source-document verification.

> **Important:** This notebook does not claim that aggregate national demand validates feeder-level control. It also does not invent the game-theory coefficients from PDF Section 1.2.

## Part 1 — Workbook audit, target selection, and leakage-safe forecasting

We first inspect the workbook instead of assuming its schema. Select a sheet and target only after reviewing the printed inventory. The split is chronological: earlier observations train the model, the middle segment selects it, and the final segment is held back for one final evaluation.

The forecasting task is defined as **one-step-ahead prediction**. For a different sampling interval or prediction horizon, update the configuration and re-run the audit.

In [ ]:
# Install if needed:
# %pip install pandas openpyxl scikit-learn tensorflow matplotlib

from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score,
    confusion_matrix, classification_report
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Project configuration: edit these after inspecting the workbook ----
WORKBOOK_PATH = Path("Qualcomm_PS10_Energy_Dataset.xlsx")
SHEET_NAME = None                 # None = inspect all sheets first
TIMESTAMP_COL = None              # e.g. "Date"
TARGET_COL = None                 # e.g. "Demand_MW"
NODE_COL = None                   # set only if workbook has real node/feeder IDs
SOURCE_LABEL = "workbook-derived"
FORECAST_HORIZON_STEPS = 1
SEASONAL_PERIOD = 24              # use 24 only for hourly data; otherwise change
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

In [ ]:
assert WORKBOOK_PATH.exists(), (
    f"Workbook not found: {WORKBOOK_PATH.resolve()}\n"
    "Place the Excel workbook beside this notebook or update WORKBOOK_PATH."
)

xls = pd.ExcelFile(WORKBOOK_PATH)
print("Workbook:", WORKBOOK_PATH.name)
print("Sheets:", xls.sheet_names)

sheet_inventory = []
for sheet in xls.sheet_names:
    frame = pd.read_excel(WORKBOOK_PATH, sheet_name=sheet)
    sheet_inventory.append({
        "sheet": sheet,
        "rows": len(frame),
        "columns": len(frame.columns),
        "column_names": list(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "duplicate_rows": int(frame.duplicated().sum())
    })

pd.DataFrame(sheet_inventory)

In [ ]:
# Inspect the selected sheet.
if SHEET_NAME is None:
    SHEET_NAME = xls.sheet_names[0]
raw = pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME)

print("Selected sheet:", SHEET_NAME)
display(raw.head())
display(pd.DataFrame({
    "column": raw.columns,
    "dtype": [str(raw[c].dtype) for c in raw.columns],
    "missing": [int(raw[c].isna().sum()) for c in raw.columns],
    "unique": [int(raw[c].nunique(dropna=True)) for c in raw.columns]
}))

### Schema confirmation

Do not proceed until `TIMESTAMP_COL` and `TARGET_COL` are confirmed. Confirm the target's **meaning and unit** from the workbook/source documentation. If the sheet contains several regions or feeders, select the relevant series deliberately; do not silently combine them.

In [ ]:
# Set these after inspecting the displayed columns:
# TIMESTAMP_COL = "..."
# TARGET_COL = "..."

if TIMESTAMP_COL is None or TARGET_COL is None:
    print("PAUSE: set TIMESTAMP_COL and TARGET_COL after reviewing the schema.")
else:
    data = raw[[TIMESTAMP_COL, TARGET_COL]].copy()
    data[TIMESTAMP_COL] = pd.to_datetime(data[TIMESTAMP_COL], errors="coerce")
    data[TARGET_COL] = pd.to_numeric(data[TARGET_COL], errors="coerce")
    data = data.dropna().sort_values(TIMESTAMP_COL).reset_index(drop=True)

    print("Rows after required-field cleaning:", len(data))
    print("Duplicate timestamps:", int(data[TIMESTAMP_COL].duplicated().sum()))
    print("Time range:", data[TIMESTAMP_COL].min(), "to", data[TIMESTAMP_COL].max())
    print("Target summary:")
    display(data[TARGET_COL].describe())

    gaps = data[TIMESTAMP_COL].diff().value_counts().head(10)
    print("Most common timestamp intervals:")
    display(gaps)

In [ ]:
# Feature engineering: all features use information available at prediction time.
# This cell runs only after schema configuration.
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    data = data.copy()
    y = data[TARGET_COL].astype(float)

    data["lag_1"] = y.shift(1)
    data["lag_2"] = y.shift(2)
    data["lag_3"] = y.shift(3)
    data["rolling_mean_3"] = y.shift(1).rolling(3).mean()
    data["rolling_std_3"] = y.shift(1).rolling(3).std()
    data["hour"] = data[TIMESTAMP_COL].dt.hour
    data["dayofweek"] = data[TIMESTAMP_COL].dt.dayofweek
    data["month"] = data[TIMESTAMP_COL].dt.month

    # Cyclic encodings avoid treating hour 23 and hour 0 as far apart.
    data["hour_sin"] = np.sin(2*np.pi*data["hour"]/24)
    data["hour_cos"] = np.cos(2*np.pi*data["hour"]/24)
    data["dow_sin"] = np.sin(2*np.pi*data["dayofweek"]/7)
    data["dow_cos"] = np.cos(2*np.pi*data["dayofweek"]/7)

    FEATURE_COLS = [
        "lag_1", "lag_2", "lag_3", "rolling_mean_3", "rolling_std_3",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos"
    ]
    supervised = data.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
    X = supervised[FEATURE_COLS].astype(np.float32).to_numpy()
    y = supervised[TARGET_COL].astype(np.float32).to_numpy()
    timestamps = supervised[TIMESTAMP_COL].to_numpy()

    print("Supervised samples:", len(X))
    print("Features:", FEATURE_COLS)

### Chronological split and baseline models

The final test segment is not used for feature selection, threshold selection, or early stopping. Persistence and seasonal-naive baselines are included because a small neural network is only useful if it improves on simple, credible alternatives.

In [ ]:
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    n = len(X)
    train_end = int(n * TRAIN_FRAC)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

    X_train, y_train = X[:train_end], y[:train_end]
    X_val, y_val = X[train_end:val_end], y[train_end:val_end]
    X_test, y_test = X[val_end:], y[val_end:]
    ts_test = timestamps[val_end:]

    # Fit preprocessing on train only.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)

    # Persistence: next value = latest observed value.
    pred_persistence = X_test[:, FEATURE_COLS.index("lag_1")]

    # Seasonal naive: use the value from one seasonal period ago.
    # For hourly data, this requires an additional lag_24 feature.
    # If unavailable, report as not computed rather than fabricate it.
    def regression_metrics(y_true, y_pred):
        return {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "R2": r2_score(y_true, y_pred)
        }

    baseline_results = {
        "Persistence": regression_metrics(y_test, pred_persistence)
    }
    pd.DataFrame(baseline_results).T

In [ ]:
# Compact MLP — intentionally small.
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1)
    ], name="cooperative_grid_tiny_mlp")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=15, restore_best_weights=True
        )
    ]

    history = model.fit(
        X_train_s, y_train,
        validation_data=(X_val_s, y_val),
        epochs=200,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )

    pred_val = model.predict(X_val_s, verbose=0).ravel()
    pred_test = model.predict(X_test_s, verbose=0).ravel()

    results = {
        "Persistence": regression_metrics(y_test, pred_persistence),
        "TinyML Float32": regression_metrics(y_test, pred_test)
    }
    results_df = pd.DataFrame(results).T
    display(results_df)

    plt.figure(figsize=(12, 4))
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="validation")
    plt.title("Training curve")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()

### Operational event classification: overload-risk precision, recall, and F1

Forecasting metrics are regression metrics. Precision/recall/F1 require a defined event label. Here, the example label is **actual target demand above a configured operating limit**. Set `OPERATING_LIMIT` from an approved source; do not infer it from the test set.

The classification threshold is applied to the forecasted demand. This is a simple event detector, not a protection relay and not a substitute for electrical feasibility analysis.

In [ ]:
# Configure from an approved operating limit or project scenario.
OPERATING_LIMIT = None  # e.g. 950.0 in the same unit as TARGET_COL

if TIMESTAMP_COL is not None and TARGET_COL is not None and OPERATING_LIMIT is not None:
    y_event = (y_test > OPERATING_LIMIT).astype(int)
    pred_event = (pred_test > OPERATING_LIMIT).astype(int)

    event_metrics = {
        "Precision": precision_score(y_event, pred_event, zero_division=0),
        "Recall": recall_score(y_event, pred_event, zero_division=0),
        "F1": f1_score(y_event, pred_event, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_event, pred_event)
    }
    display(pd.Series(event_metrics, name="Overload-risk event metrics"))

    print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrix(y_event, pred_event))
    print(classification_report(y_event, pred_event, zero_division=0))
else:
    print("Event metrics not computed: configure OPERATING_LIMIT from an approved source.")

In [ ]:
# Error inspection: highest absolute errors on held-out test data.
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    error_table = pd.DataFrame({
        "timestamp": ts_test,
        "actual": y_test,
        "prediction": pred_test,
        "absolute_error": np.abs(y_test - pred_test)
    }).sort_values("absolute_error", ascending=False)

    display(error_table.head(15))

    plt.figure(figsize=(12, 4))
    plt.plot(ts_test, y_test, label="actual")
    plt.plot(ts_test, pred_test, label="TinyML")
    plt.title("Held-out test: actual vs predicted")
    plt.xlabel("Time")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## Part 2 — One shared model, four dynamically configured node agents

The same model is reused for N1–N4. Each node has its own data/history and operating configuration. This notebook supports two modes:

- **Workbook-backed:** use real node/feeder series only if the workbook actually contains a node identifier and corresponding target observations.
- **Scenario simulation:** generate clearly labelled node inputs for testing the software path. These are not real feeder measurements and must not be used to claim model generalisation.

The inference output is shaped for the backend and frontend. The Raspberry Pi coordinator can consume these records; the ESP32 should receive compact decisions, not raw workbook rows.

In [ ]:
# A shared inference function. The node identity does not alter model weights.
MODEL_VERSION = "cg-tiny-mlp-v1"

def predict_node(node_id, feature_row, source_label="workbook-derived"):
    feature_row = np.asarray(feature_row, dtype=np.float32).reshape(1, -1)
    feature_scaled = scaler.transform(feature_row).astype(np.float32)
    forecast = float(model.predict(feature_scaled, verbose=0).ravel()[0])

    return {
        "node_id": str(node_id),
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "forecast": forecast,
        "model_version": MODEL_VERSION,
        "source": source_label
    }

# Demonstration using the last available feature row for each configured node.
# Replace this with node-specific history when genuine node data are available.
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    example_nodes = ["N1", "N2", "N3", "N4"]
    latest_feature_row = X[-1]

    node_predictions = [
        predict_node(node_id, latest_feature_row, source_label=SOURCE_LABEL)
        for node_id in example_nodes
    ]
    display(pd.DataFrame(node_predictions))

In [ ]:
# Example API-compatible telemetry envelope.
# Values below are placeholders unless populated from real approved measurements.
def telemetry_payload(node_id, observed_load, forecast_load, limit, flexible_load,
                      source="simulated", sequence=1, boot_id="demo-boot"):
    return {
        "schema_version": "1.0",
        "node_id": node_id,
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "sequence": int(sequence),
        "boot_id": boot_id,
        "observed_load_kw": float(observed_load),
        "forecast_load_kw": float(forecast_load),
        "limit_kw": float(limit),
        "flexible_load_kw": float(flexible_load),
        "source": source,
        "model_version": MODEL_VERSION
    }

# Example only — do not treat as measured data.
example_payload = telemetry_payload(
    node_id="N2",
    observed_load=820,
    forecast_load=910,
    limit=950,
    flexible_load=120,
    source="simulated"
)
print(json.dumps(example_payload, indent=2))

## Part 3 — Cooperative decision baseline, game-theory integration, and INT8 deployment

This section deliberately separates three things:

1. **Decision baseline:** a transparent coordinator that can compare candidate actions.
2. **PDF game formulation:** exact α, β, γ, δ definitions must be transcribed from Section 1.2 before implementation.
3. **Electrical feasibility:** a proposed reallocation must be checked against an approved network model (for example, OpenDSS) before it is represented as feasible.

The example below is only a software-interface placeholder. It does not claim to implement the paper’s exact utility, Shapley value, Stackelberg game, or trained MARL policy.

In [ ]:
# ---- Game-theory source transcription gate ----
GAME_FORMULA_SOURCE = "PDF Section 1.2 — pending exact transcription"
ALPHA = None
BETA = None
GAMMA = None
DELTA = None

def validate_game_parameters():
    params = {"alpha": ALPHA, "beta": BETA, "gamma": GAMMA, "delta": DELTA}
    missing = [k for k, v in params.items() if v is None]
    if missing:
        raise ValueError(
            "Do not run the PDF-specific reward yet. Missing exact source values/definitions: "
            + ", ".join(missing)
        )
    return params

print("Game reward locked until exact PDF definitions are entered.")

In [ ]:
# Transparent placeholder for integration testing only.
# Replace with the verified Section 1.2 objective and electrical constraints.
def coordinator_placeholder(node_records):
    proposals = []
    for rec in node_records:
        predicted = rec["forecast_load_kw"]
        limit = rec["limit_kw"]
        flexible = rec["flexible_load_kw"]
        risk = max(0.0, predicted - limit)
        suggested_reduction = min(risk, flexible)

        proposals.append({
            "node_id": rec["node_id"],
            "risk_kw": risk,
            "suggested_reduction_kw": suggested_reduction,
            "decision_status": "PROPOSAL_ONLY_NOT_ELECTRICALLY_VALIDATED"
        })
    return proposals

# This function intentionally does not transfer load between nodes.
# Transfer requires topology, line/transformer limits, voltage constraints,
# source/sink availability, protection/interlocking, and an electrical solver.

### INT8 conversion and deployment checks

The representative dataset is drawn only from training features. Quantisation is evaluated against the float model on the held-out test inputs. Conversion success alone does not establish ESP32 compatibility: operator support, memory arena, latency, and the exact target runtime must still be checked on the selected board.

In [ ]:
# Save the Float32 model and preprocessing manifest.
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

if TIMESTAMP_COL is not None and TARGET_COL is not None:
    model.save(ARTIFACT_DIR / "cooperative_grid_float32.keras")

    manifest = {
        "model_version": MODEL_VERSION,
        "target_column": TARGET_COL,
        "timestamp_column": TIMESTAMP_COL,
        "feature_columns": FEATURE_COLS,
        "source_label": SOURCE_LABEL,
        "forecast_horizon_steps": FORECAST_HORIZON_STEPS,
        "split": {
            "train_fraction": TRAIN_FRAC,
            "validation_fraction": VAL_FRAC,
            "test_fraction": round(1 - TRAIN_FRAC - VAL_FRAC, 4),
            "chronological": True
        },
        "note": "Confirm target units and source provenance before deployment."
    }
    with open(ARTIFACT_DIR / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    np.savez(
        ARTIFACT_DIR / "scaler.npz",
        mean=scaler.mean_,
        scale=scaler.scale_
    )

In [ ]:
# Full integer quantisation to TFLite.
# Representative data: training set only.
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    def representative_dataset():
        count = min(300, len(X_train_s))
        for row in X_train_s[:count]:
            yield [row.reshape(1, -1).astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    tflite_path = ARTIFACT_DIR / "cooperative_grid_int8.tflite"
    tflite_path.write_bytes(tflite_model)

    print("TFLite INT8 size (bytes):", tflite_path.stat().st_size)

In [ ]:
# Evaluate quantised model parity and host inference latency.
if TIMESTAMP_COL is not None and TARGET_COL is not None:
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]

    def quantize_input(x_float, detail):
        scale, zero_point = detail["quantization"]
        if scale == 0:
            raise ValueError("Invalid input quantisation scale.")
        q = np.round(x_float / scale + zero_point)
        return np.clip(q, -128, 127).astype(np.int8)

    def dequantize_output(q, detail):
        scale, zero_point = detail["quantization"]
        return (q.astype(np.float32) - zero_point) * scale

    n_eval = min(500, len(X_test_s))
    float_preds = model.predict(X_test_s[:n_eval], verbose=0).ravel()
    int8_preds = []

    start = time.perf_counter()
    for row in X_test_s[:n_eval]:
        q_in = quantize_input(row.reshape(1, -1), input_info)
        interpreter.set_tensor(input_info["index"], q_in)
        interpreter.invoke()
        q_out = interpreter.get_tensor(output_info["index"])
        int8_preds.append(float(dequantize_output(q_out, output_info).ravel()[0]))
    elapsed = time.perf_counter() - start

    int8_preds = np.asarray(int8_preds)
    parity = {
        "Float32 MAE": mean_absolute_error(y_test[:n_eval], float_preds),
        "INT8 MAE": mean_absolute_error(y_test[:n_eval], int8_preds),
        "Mean absolute Float32-INT8 difference": float(np.mean(np.abs(float_preds-int8_preds))),
        "Host latency ms/sample": 1000 * elapsed / n_eval,
        "INT8 model size bytes": int(tflite_path.stat().st_size)
    }
    display(pd.Series(parity))

### Final experiment record

Before presenting results, record:
- workbook filename, sheet, target, unit, and source provenance;
- exact time interval and prediction horizon;
- chronological split boundaries;
- baseline and TinyML metrics;
- overload threshold source and event metrics;
- INT8 size, host latency, and float/INT8 parity;
- whether four-node inputs are workbook-backed or simulated;
- exact PDF Section 1.2 formula source and electrical-feasibility status.

**Do not report simulated node outputs as field measurements. Do not report host latency as ESP32 latency. Do not report a proposed transfer as electrically feasible until the network model validates it.**